In [1]:
DEBUG = 1

### **Mount to the Drive**



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
base_dir = '/content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2'
dataset_dir = f'{base_dir}/data/kitti_sample'
pointcloud_output_path = f"{base_dir}/outputs/kitti_tsdf_global_map.ply"
mesh_output_path = f"{base_dir}/outputs/kitti_tsdf_solid_mesh.ply"

%cd $base_dir

/content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2


In [4]:
# Run the command below if you need
!pip install -q -r requirements/visual_odometry.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 MB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 36.5 MB/s eta 0:00:00


In [5]:
import os
import cv2 as cv
import numpy as np
import open3d as o3d
import ipywidgets as widgets
from ultralytics import YOLO
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from src.export import export_all_layers
from src.data_loader import KittiDataset
from src.geometry import reprojectImageTo3D
from IPython.display import display, clear_output
from src.visualization import visualize_odometry_scene
from src.feature_extraction import get_3d_bounding_boxes

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [6]:
intrinsic = None
p0 = None
global_t = np.zeros((3, 1))
global_R = np.eye(3)
trajectory = []

In [7]:
lk_params = {
    'winSize': (21, 21),
    'maxLevel': 3,
    'criteria': (cv.TERM_CRITERIA_EPS | cv.TERM_CRITERIA_COUNT, 30, 0.01)
    }

In [8]:
kittiDataset = KittiDataset(dataset_dir)

In [9]:
K = kittiDataset.get_projection_2()[:,:3]
MIN_FEATURE_COUNT = 500

In [10]:
yolo_model = YOLO("yolo26n.pt")

In [11]:
num_of_tracked_pts = []
global_3d_boxes = []
point_id_counter = 0
point_ids = np.array([], dtype=np.int32)
total_frames = kittiDataset.get_total_frames()
global_pcd = o3d.geometry.PointCloud()

image_widget = widgets.Image(format='jpeg', width=600, height=300)
log_widget = widgets.Output(layout=widgets.Layout(max_height='150px', overflow='y-scroll'))

video_output_path = f"{base_dir}/outputs/features.mp4"
video_writer = None

recent_logs = []

def log_message(msg, max_lines=8):
    recent_logs.insert(0, msg)
    if len(recent_logs) > max_lines:
        recent_logs.pop()

    with log_widget:
        clear_output(wait=True)
        print("\n".join(recent_logs))

if DEBUG:
    display(widgets.VBox([image_widget, log_widget]))

with tqdm(total=total_frames, desc="KITTI Odometry", unit="frame") as pbar:
    while not kittiDataset.is_done():
        left_img, right_img, next_left_img = kittiDataset.get_next_frame()

        left_img_gray = cv.cvtColor(left_img, cv.COLOR_BGR2GRAY)
        next_left_img_gray = cv.cvtColor(next_left_img, cv.COLOR_BGR2GRAY)
        h, w, c = left_img.shape

        if p0 is None or len(p0) < MIN_FEATURE_COUNT:
            new_p0  = cv.goodFeaturesToTrack(left_img_gray, 2000, 0.01, 10)
            if new_p0 is not None:
                num_new_pts = len(new_p0)
                new_ids = np.arange(point_id_counter, point_id_counter + num_new_pts, dtype=np.int32)
                point_id_counter += num_new_pts

                if p0 is None:
                    p0 = new_p0
                    point_ids = new_ids
                else:
                    p0 = np.vstack((p0, new_p0))
                    point_ids = np.concatenate((point_ids, new_ids))

            num_of_tracked_pts.append([kittiDataset.get_idx(), len(p0)])

            log_message(f"Frame {kittiDataset.get_idx()}: Insufficient features! Extracted {len(p0)} new features.")

        reconstruction, disparity = reprojectImageTo3D(left_img, right_img, kittiDataset.Q)
        p1, status, err = cv.calcOpticalFlowPyrLK(left_img_gray, next_left_img_gray, p0, None, **lk_params)

        flow_status = (status == 1).reshape(-1)
        good_old = p0[flow_status].reshape(-1, 2)
        good_new = p1[flow_status].reshape(-1, 2)
        good_ids = point_ids[flow_status]

        object_points_3d, image_points_2d = [], []
        valid_mask = []

        for old, new in zip(good_old, good_new):
            old_x, old_y = int(round(old[0])), int(round(old[1]))
            new_x, new_y = int(round(new[0])), int(round(new[1]))
            is_valid = False

            if 0 <= old_y < h and 0 <= old_x < w:
                if disparity[old_y, old_x] > 0:
                    point_3d = reconstruction[old_y, old_x]
                    if 0.1 < point_3d[2] < 50.0:
                        object_points_3d.append(point_3d)
                        image_points_2d.append([new_x, new_y])
                        is_valid = True
            valid_mask.append(is_valid)

        object_points_3d = np.array(object_points_3d, dtype=np.float32)
        image_points_2d = np.array(image_points_2d, dtype=np.float32)
        valid_ids = good_ids[np.array(valid_mask)]

        if len(object_points_3d) >= 4:
            success, rvec, tvec, inliers = cv.solvePnPRansac(
                objectPoints=object_points_3d, imagePoints=image_points_2d,
                cameraMatrix=K, distCoeffs=None, flags=cv.SOLVEPNP_ITERATIVE
            )
        else:
            success = False
            inliers = None
            log_message(f"Frame {kittiDataset.get_idx()}: Passed PnP step! Insufficient 3D points ({len(object_points_3d)}).")

        inlier_count = len(inliers) if (success and inliers is not None) else 0

        vis_img = left_img.copy() if DEBUG else None

        if success:
            R_mat, _ = cv.Rodrigues(rvec)
            global_t = global_t - global_R @ R_mat.T @ tvec
            global_R = global_R @ R_mat.T
            trajectory.append(global_t.flatten())

            rgb_clean = cv.cvtColor(left_img, cv.COLOR_BGR2RGB)

            valid_mask = (disparity > 0) & (reconstruction[:, :, 2] > 0.1) & (reconstruction[:, :, 2] < 50.0)

            points_3d = reconstruction[valid_mask]
            colors_3d = rgb_clean[valid_mask] / 255.0

            current_pcd = o3d.geometry.PointCloud()
            current_pcd.points = o3d.utility.Vector3dVector(points_3d)
            current_pcd.colors = o3d.utility.Vector3dVector(colors_3d)

            pose = np.eye(4)
            pose[:3, :3] = global_R
            pose[:3, 3] = global_t.flatten()
            current_pcd.transform(pose)

            global_pcd += current_pcd
            global_pcd = global_pcd.voxel_down_sample(voxel_size=0.1)

            log_message(f"Frame {kittiDataset.get_idx()}: Global PointCloud updated. ({len(global_pcd.points)} points)")

            if kittiDataset.get_idx() % 5 == 0:
                depth_map = reconstruction[:, :, 2].copy()
                depth_map = np.nan_to_num(depth_map, nan=0.0, posinf=0.0, neginf=0.0)
                new_3d_boxes = get_3d_bounding_boxes(yolo_model, left_img, depth_map, K, global_R, global_t)
                global_3d_boxes.extend(new_3d_boxes)

        else:
            log_message(f"Frame {kittiDataset.get_idx()}: PnP RANSAC failed!")

        if success and inliers is not None:
            inlier_indices = inliers.flatten()
            point_ids = valid_ids[inlier_indices]
            p0 = image_points_2d[inlier_indices].reshape(-1, 1, 2)
        else:
            p0 = None
            point_ids = np.array([], dtype=np.int32)

        if DEBUG:
            vis_img = cv.resize(vis_img, (w * 2, h * 2), interpolation=cv.INTER_LINEAR)

            for pt, p_id in zip(image_points_2d, point_ids):
                pt_x, pt_y = int(pt[0]) * 2, int(pt[1]) * 2
                cv.circle(vis_img, (pt_x, pt_y), 4, (0, 255, 0), -1)
                cv.putText(vis_img, str(p_id), (pt_x + 5, pt_y - 5),
                           cv.FONT_HERSHEY_SIMPLEX, 0.4, (0, 0, 0), 2)
                cv.putText(vis_img, str(p_id), (pt_x + 5, pt_y - 5),
                           cv.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 255), 1)

            text = f"Frame {kittiDataset.get_idx()} | Features: {len(image_points_2d)}"
            (text_width, text_height), _ = cv.getTextSize(text, cv.FONT_HERSHEY_SIMPLEX, 0.8, 2)
            pad_x, pad_y_top, pad_y_bottom = 15, 10, 11
            top_left = (10, 10)
            bottom_right = (10 + text_width + pad_x * 2, 10 + text_height + pad_y_bottom * 2)

            cv.rectangle(vis_img, top_left, bottom_right, (88, 38, 2), -1)
            cv.putText(vis_img, text, (10 + pad_x, 10 + text_height + pad_y_top),
                       cv.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

            if video_writer is None:
                fourcc = cv.VideoWriter_fourcc(*'mp4v')
                video_writer = cv.VideoWriter(video_output_path, fourcc, 10.0, (w * 2, h * 2))

            video_writer.write(vis_img)

            _, encoded_img = cv.imencode('.jpg', vis_img)
            image_widget.value = encoded_img.tobytes()

        pbar.set_postfix({
            "Frame": kittiDataset.get_idx(),
            "Inliers": inlier_count
        })

        pbar.update(1)

    if video_writer is not None:
        video_writer.release()
        print(f"Video have been successfully saved to: {video_output_path}")

KITTI Odometry:   0%|          | 0/600 [00:00<?, ?frame/s]

Video have been successfully saved to: /content/drive/MyDrive/epipolar-geometry-dinov2/epipolar-geometry-dinov2/outputs/features.mp4


In [12]:
export_all_layers(global_pcd, trajectory, global_3d_boxes, base_dir)

Statistical Outlier Removal...
Saving Point Cloud...
Saving trajectory...
Filtering and saving YOLO 3D bounding boxes...
All layers have been successfully saved!


In [14]:
visualize_odometry_scene(global_pcd, trajectory, global_3d_boxes)

Output hidden; open in https://colab.research.google.com to view.